# Repo-owned v586 extraction scaffold

This notebook is a repo-owned frozen extraction scaffold from `claudedevore/birdclef-2026-r0946-a2prime-effv2s-submit` v5.

Purpose: prepare the A2Prime EfficientNetV2-S branch as a fallback **only if** v585 FrankSunP fails/drops and no stronger `0.950+` source appears.

Why this branch was selected over NFNet:

- public audit found the EffV2S branch schema-safe on sample output;
- `a2prime_blend_summary.csv` reported Proto/EffV2S rank correlation ≈ `0.053`, lower than the NFNet branch's ≈ `0.169`;
- EffV2S sanity top-5 hit rate was `0.55`, higher than NFNet's `0.30`;
- v581 already no-scored on an A2Prime/NFNet replay, so NFNet is not the first fallback.

Safety policy:

- Do not push or submit this kernel while `birdclef-v585-reset` exists and owns the next reset slot.
- Before any slot use, run the push helper, verify COMPLETE/no failure, and verify `submission.csv` is competition-shaped, finite, non-constant, and sample-aligned.
- If v585 improves above `0.949`, abandon this fallback and port/confirm FrankSunP instead.


# 🐦 BirdCLEF+ 2026 — Visual Explained Pipeline

This notebook is a **readable / public-notebook-ready rewrite** of the original BirdCLEF+ 2026 pipeline.

## What was preserved
- The original execution order and model logic are preserved.
- ProtoSSM, ResidualSSM, SED, BirdNET, rank blending, and all post-processing gates remain in the same functional flow.
- This version mainly adds **cell structure, visual explanations, checklists, and final diagnostics**.

## Original note
# BirdCLEF+ 2026 — v4: Exact 0.946 Pipeline + BirdNET Third Branch

**Base:** Exact copy of Imaad Mahmood's 0.946 pipeline (all params preserved).
**Addition:** BirdNET v2.4 TFLite as a third independent model branch.

**What is EXACTLY preserved from the 0.946 original:**
- `correction_weight=0.30` (hardcoded in call, not from CFG)
- `train_residual_ssm(n_epochs=30, patience=8)` (hardcoded)
- `train_light_proto_ssm(n_epochs=40, patience=8)` (hardcoded)
- `ENSEMBLE_W=0.5`, `alpha_blend=0.4`, `sigma=0.65`
- All 5 blend gates with original thresholds
- 60/40 ProtoSSM/SED base blend

**What is new:**
- BirdNET v2.4 (TFLite, 6522 species) run as third branch
- If BirdNET available: 50/30/20 Proto/SED/BirdNET rank blend
- If BirdNET unavailable: falls back to original 60/40 blend exactly
- New Gate 3b: BirdNET spike preservation

**Attribution:** Vyanktesh Dwivedi (base), Imaad Mahmood (ProtoSSM), Tucker Arrants (SED),
Stefan Kahl et al. (BirdNET), Shadi Akiki (BirdNET Kaggle model)

## 0. End-to-end visual flow

```text
Raw 60s OGG files
      │
      ├──► Perch / Bird Vocalization Classifier
      │       ├── Competition-aligned logits
      │       └── 1536-d embeddings
      │
      ├──► ProtoSSM sequence model
      │       └── 12-window temporal predictions
      │
      ├──► Ecological priors + MLP probes
      │       ├── Site prior
      │       ├── Hour prior
      │       └── Site × Hour prior
      │
      ├──► ResidualSSM correction
      │       └── Learns residual correction after first pass
      │
      ├──► Distilled SED branch
      │       └── Mel spectrogram + ONNX fold ensemble
      │
      ├──► BirdNET branch, optional
      │       └── 3s chunks mapped back to 5s competition windows
      │
      └──► Rank blend + gates
              ├── Noise suppression
              ├── Temporal continuity
              ├── SED spike preservation
              ├── BirdNET spike preservation
              ├── Sonotype mirroring
              └── Rare-class thresholding

Final output: submission.csv
```

## 1. Pipeline stage map

| Stage | Input | Output | Why it matters |
|---|---|---|---|
| Perch feature extraction | 60s audio split into 12 × 5s windows | logits + embeddings | Strong general acoustic backbone |
| Cache | Perch outputs | reusable parquet / npz | Avoids recomputing expensive features |
| ProtoSSM | Perch embeddings + logits + metadata | temporal logits | Learns sequence context across 12 windows |
| Priors | site / hour / site-hour buckets | logit correction | Adds ecological probability context |
| MLP probes | PCA embeddings + score dynamics | refined logits | Class-specific correction layer |
| ResidualSSM | first-pass logits + embeddings | residual correction | Fixes systematic first-pass errors |
| SED branch | mel spectrograms | window probabilities | Captures short acoustic events |
| BirdNET branch | 3s audio chunks | optional window probabilities | Adds independent species detector signal |
| Final blend | ProtoSSM + SED + BirdNET | submission.csv | Rank blending plus safety gates |

## 2. Execution checklist

Before running on Kaggle, attach the required public datasets/models used by the original notebook:

| Asset | Used for |
|---|---|
| `birdclef-2026` competition data | taxonomy, labels, test audio, sample submission |
| `jaejohn/perch-meta` | optional external Perch cache |
| `rishikeshjani/perch-onnx-for-birdclef-2026` | ONNX Runtime wheel / Perch ONNX support |
| `tuckerarrants/bc2026-distilled-sed-public` | SED fold ONNX models |
| `tuckerarrants/perch-v2-no-dft-onnx` | optional fast Perch ONNX model |
| `ashok205/tf-wheels` | TensorFlow 2.20 offline wheels |
| `google/bird-vocalization-classifier` | Perch SavedModel fallback |
| BirdNET TFLite model, optional | third branch; skipped safely if absent |

The notebook is designed to run in `MODE = "submit"` by default.

---
## Cell 01 — Environment, package installation, seeds, mode, CFG

This cell prepares the offline Kaggle runtime. It installs local wheels when available, fixes random seeds, disables GPU for CPU-safe inference, and defines the global configuration. The model logic is not changed.

In [ ]:
import subprocess, sys, os
from pathlib import Path
import random
import numpy as np
import torch

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(4)
print("Global random seed set to 4")

MODE = "submit"
assert MODE in {"train", "submit"}
print("MODE =", MODE)

import os, re, gc, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

tf.experimental.numpy.experimental_enable_numpy_behavior()
try: tf.config.set_visible_devices([], "GPU")
except: pass

_WALL_START = time.time()

BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12

CFG = {
    "batch_files": 16,
    "oof_n_splits": 5   if MODE == "train" else 3,
    "dryrun_n_files": 20 if MODE == "train" else 0,
    "run_oof": MODE == "train",
    "verbose": MODE == "train",
    "proto_ssm_train": {
        "n_epochs":        80  if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        20  if MODE == "train" else 8,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5   if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
    },
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,
        "n_epochs": 40  if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12  if MODE == "train" else 6,
    },
    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500  if MODE == "train" else 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20  if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },
}
print("CFG loaded")


---
## Cell 02 — Data

Loads taxonomy, sample submission, and soundscape labels. It converts window-level labels into a multi-label target matrix and keeps only fully labeled 60-second files with 12 five-second windows.

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)

_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")


---
## Cell 03 — Perch backbone

Loads Google Perch, maps Perch/Bird Vocalization logits to BirdCLEF competition labels, and builds genus-level proxy signals for unmapped species.

In [ ]:
# ── Perch backbone ────────────────────────────────────────────────────────────
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

# Prefer no-DFT variant, fallback to standard
ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                   next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path("")))
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print(f"Using ONNX Perch: {ONNX_PERCH_PATH.name}")
else:
    print("Using TF SavedModel Perch")

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

import re as _re
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

proxy_map = {}
unmapped_df = (taxonomy[taxonomy["primary_label"]
               .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy())

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(rf"^{_re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | No signal: {len(UNMAPPED_POS)-len(proxy_map)}")


---
## Cell 04 — Per-taxon temperatures

Defines class-wise temperature scaling. Texture-like taxa receive a slightly sharper calibration than typical bird classes.

In [ ]:
# ── Per-taxon temperatures ────────────────────────────────────────────────────
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10


---
## Cell 05 — Perch inference engine

Reads each 60-second audio file, slices it into 12 windows, runs Perch, and returns row_ids, metadata, raw logits, and 1536-dimensional embeddings.

In [ ]:
# ── Perch inference engine ────────────────────────────────────────────────────
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)
    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr
            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS
            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb
            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
            del x, logits, emb, batch_audio
            gc.collect()
    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("Perch inference engine defined")


---
## Cell 06 — Cache

Uses an external Perch cache when available; otherwise builds a local cache from the training soundscapes. This is the heavy feature-extraction checkpoint.

In [ ]:
# ── Cache ─────────────────────────────────────────────────────────────────────
print(f"USE_ONNX = {USE_ONNX}")

EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]
CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"

def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None

SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]

def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")

def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")
    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]
    t0 = time.time()
    meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=CFG["batch_files"], verbose=True)
    print(f"  Perch pass done in {time.time()-t0:.1f}s  scores={sc_built.shape} embs={emb_built.shape}")
    meta_built.to_parquet(CACHE_META_LOCAL)
    np.savez(CACHE_NPZ_LOCAL, scores=sc_built.astype(np.float32),
             embs=emb_built.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))
    print(f"  Cache saved to {WORK_DIR}")
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL

ext_meta, ext_npz = _find_external_cache()
if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")
else:
    print("No cache found — building from scratch")
    CACHE_META, CACHE_NPZ = _build_cache()

meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

if "primary_labels" in _arr.files:
    if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
        print("  WARNING: cached primary_labels differ — scores columns may not align!")
    else:
        print("  primary_labels schema OK")

if "row_id" not in meta_tr.columns:
    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)
    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5
    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = (meta_tr["filename"].str.replace(".ogg", "", regex=False)
                         + "_" + end_sec.astype(str))

row_id_to_index = full_rows.set_index("row_id")["index"]
missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing_rows:
    raise RuntimeError(f"Cache has {len(missing_rows)} row_ids not in labeled set.")

Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]
print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")


---
## Cell 07 — Post-processing helpers

Defines small reusable utilities: macro AUC, temporal smoothing, confidence scaling, rank-aware scaling, and adaptive smoothing.

In [ ]:
# ── Post-processing helpers ───────────────────────────────────────────────────
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def smooth_predictions(probs, n_windows=12, alpha=0.3):
    N, C = probs.shape
    assert N % n_windows == 0
    view = probs.reshape(-1, n_windows, C).copy()
    prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
    next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
    return ((1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)).reshape(N, C)


---
## Cell 08 — UPGRADED prior tables — joint site-hour bucket

Builds global, site, hour, and site-hour priors, then applies them as logit corrections. This injects ecological context without changing the core Perch/SSM architecture.

In [ ]:
# ── UPGRADED prior tables — joint site-hour bucket ────────────────────────────
def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_labels[mask].mean(axis=0)

    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_labels[mask].mean(axis=0)

    # Joint site-hour bucket (new — tighter shrinkage factor 4)
    sh_keys = sorted({(str(s), int(h)) for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())
                      if not pd.isna(s) and not pd.isna(h)})
    sh_to_i = {k: i for i, k in enumerate(sh_keys)}
    sh_p = np.zeros((len(sh_keys), Y_labels.shape[1]), dtype=np.float32)
    sh_n = np.zeros(len(sh_keys), dtype=np.float32)
    for (s, h) in sh_keys:
        i = sh_to_i[(s, h)]
        mask = (sc_df["site"].astype(str).values == s) & (sc_df["hour_utc"].astype(int).values == h)
        sh_n[i] = mask.sum()
        sh_p[i] = Y_labels[mask].mean(axis=0)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i, "site_p": site_p, "site_n": site_n,
        "hour_to_i": hour_to_i, "hour_p": hour_p, "hour_n": hour_n,
        "sh_to_i": sh_to_i,    "sh_p": sh_p,    "sh_n": sh_n,
    }

def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    eps = 1e-4; n = len(scores); out = scores.copy()
    p = np.tile(tables["global_p"], (n, 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]; nh = tables["hour_n"][j]; w = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]; ns = tables["site_n"][j]; w = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
    if "sh_to_i" in tables:
        for i, (s, h) in enumerate(zip(sites, hours)):
            key = (str(s), int(h))
            if key in tables["sh_to_i"]:
                j = tables["sh_to_i"][key]; nsh = tables["sh_n"][j]; w = nsh / (nsh + 4.0)
                p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    N, C = probs.shape
    view      = probs.reshape(-1, n_windows, C)
    sorted_v  = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_k_mean, power)).reshape(N, C)

def rank_aware_scaling(probs, n_windows=12, power=0.4):
    N, C = probs.shape
    view     = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    return (view * np.power(file_max, power)).reshape(N, C)

def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    N, C = probs.shape
    result = probs.copy(); view = probs.reshape(-1, n_windows, C); out = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True); alpha = base_alpha * (1.0 - conf)
        if t == 0:           neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows-1: neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:                  neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
    return result


---
## Cell 09 — MLP probes

Trains lightweight per-class MLP probes on PCA-reduced Perch embeddings plus sequential score features. The probes act as class-specific refiners before the SSM residual stage.

In [ ]:
# ── MLP probes ────────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
import torch.nn as nn
import torch.nn.functional as F

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq = pos_count / Y.shape[0]
    weights = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_sequential_features(scores_col, n_windows=12):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std

def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    scaler = StandardScaler(); emb_s = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding: {emb.shape} → PCA: {Z.shape}  (variance retained: {pca.explained_variance_ratio_.sum():.2%})")
    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models = {}; active = np.where(Y.sum(axis=0) >= min_pos)[0]; MAX_ROWS = 3000
    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1], prev[:, None], next_[:, None], mean[:, None], max_[:, None], std[:, None]])
        n_pos = int(y.sum()); n_neg = len(y) - n_pos; pos_idx = np.where(y == 1)[0]
        w = float(class_weights[ci]); repeat = max(1, min(int(round(w * n_neg / max(n_pos, 1))), 8))
        if n_pos * repeat + len(y) > MAX_ROWS: repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", max_iter=300,
                            early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
                            random_state=42, learning_rate_init=5e-4, alpha=0.005)
        clf.fit(X_bal, y_bal); probe_models[ci] = clf
    print(f"Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend

class VectorizedMLPProbes(nn.Module):
    def __init__(self, probe_models):
        super().__init__(); self.valid_classes = sorted(probe_models.keys()); V = len(self.valid_classes)
        if V == 0: self.weights = nn.ParameterList(); self.biases = nn.ParameterList(); self.n_layers = 0; return
        sample = probe_models[self.valid_classes[0]]; self.n_layers = len(sample.coefs_)
        self.weights = nn.ParameterList(); self.biases = nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))
    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1: h = torch.relu(h)
        return h.squeeze(-1)

def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4):
    if len(probe_models) == 0: return scores_test.copy()
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    valid_classes = sorted(probe_models.keys()); V = len(valid_classes); N = len(scores_test)
    raw = scores_test[:, valid_classes].T; n_files = N // N_WINDOWS; raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1); mx = np.repeat(raw_view.max(axis=2), N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_expanded.astype(np.float32), scalar_feats], axis=-1)
    vec_probe = VectorizedMLPProbes(probe_models).eval()
    with torch.no_grad(): preds = vec_probe(torch.tensor(X_all)).numpy()
    result = scores_test.copy()
    result[:, valid_classes] = (1.0 - alpha_blend) * scores_test[:, valid_classes] + alpha_blend * preds.T
    return result

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, threshold_grid=None, n_windows=12):
    if threshold_grid is None: threshold_grid = [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
    n_samples, n_cls = oof_probs.shape; thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    n_files = n_samples // n_windows
    file_oof = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y   = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
    n_calibrated = 0
    for c in range(n_cls):
        y_true = file_y[:, c]; y_prob = file_oof[:, c]
        if y_true.sum() < 3: continue
        try:
            ir = IsotonicRegression(out_of_bounds="clip"); ir.fit(y_prob, y_true); y_cal = ir.transform(y_prob)
        except: y_cal = y_prob
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp=((pred==1)&(y_true==1)).sum(); fp=((pred==1)&(y_true==0)).sum(); fn=((pred==0)&(y_true==1)).sum()
            prec=tp/(tp+fp+1e-8); rec=tp/(tp+fn+1e-8); f1=2*prec*rec/(prec+rec+1e-8)
            if f1 > best_f1: best_f1,best_t = f1,t
        thresholds[c] = best_t; n_calibrated += 1
    print(f"Calibrated {n_calibrated} classes | Mean threshold: {thresholds.mean():.3f} | Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds

def apply_per_class_thresholds(scores, thresholds):
    C = scores.shape[1]; scaled = np.copy(scores)
    for c in range(C):
        t = thresholds[c]; above = scores[:, c] > t
        scaled[above, c]  = 0.5 + 0.5 * (scores[above, c]  - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)


---
## Cell 10 — SSM Architecture

Defines the selective state-space blocks, the ProtoSSM sequence model, and the ResidualSSM correction model. This is the main temporal reasoning part of the pipeline.

In [ ]:
# ── SSM Architecture ─────────────────────────────────────────────────────────
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__(); self.d_model=d_model; self.d_state=d_state
        self.in_proj=nn.Linear(d_model,2*d_model,bias=False)
        self.conv1d=nn.Conv1d(d_model,d_model,d_conv,padding=d_conv-1,groups=d_model)
        self.dt_proj=nn.Linear(d_model,d_model,bias=True)
        A=torch.arange(1,d_state+1,dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
        self.A_log=nn.Parameter(torch.log(A)); self.D=nn.Parameter(torch.ones(d_model))
        self.B_proj=nn.Linear(d_model,d_state,bias=False); self.C_proj=nn.Linear(d_model,d_state,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x):
        B_sz,T,D=x.shape; xz=self.in_proj(x); x_ssm,z=xz.chunk(2,dim=-1)
        x_conv=F.silu(self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2))
        dt=F.softplus(self.dt_proj(x_conv)); A=-torch.exp(self.A_log)
        B=self.B_proj(x_conv); C=self.C_proj(x_conv)
        h=torch.zeros(B_sz,D,self.d_state,device=x.device); ys=[]
        for t in range(T):
            dA=torch.exp(A[None]*dt[:,t,:,None]); dB=dt[:,t,:,None]*B[:,t,None,:]
            h=h*dA+x[:,t,:,None]*dB; ys.append((h*C[:,t,None,:]).sum(-1))
        return torch.stack(ys,dim=1)+x*self.D[None,None,:]

class LightProtoSSM(nn.Module):
    def __init__(self,d_input=1536,d_model=128,d_state=16,n_classes=234,n_windows=12,
                 dropout=0.15,n_sites=20,meta_dim=16,use_cross_attn=True,cross_attn_heads=2):
        super().__init__(); self.n_classes=n_classes; self.n_windows=n_windows; self.use_cross_attn=use_cross_attn
        self.input_proj=nn.Sequential(nn.Linear(d_input,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.ssm_fwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_bwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_merge=nn.ModuleList([nn.Linear(2*d_model,d_model) for _ in range(2)])
        self.ssm_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop=nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn=nn.ModuleList([nn.MultiheadAttention(d_model,cross_attn_heads,dropout=dropout,batch_first=True) for _ in range(2)])
            self.cross_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes=nn.Parameter(torch.randn(n_classes,d_model)*0.02)
        self.proto_temp=nn.Parameter(torch.tensor(5.0))
        self.class_bias=nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha=nn.Parameter(torch.zeros(n_classes))
    def init_prototypes(self,emb_tensor,labels_tensor):
        with torch.no_grad():
            h=self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask=labels_tensor[:,c]>0.5
                if mask.sum()>0: self.prototypes.data[c]=F.normalize(h[mask].mean(0),dim=0)
    def forward(self,emb,perch_logits=None,site_ids=None,hours=None):
        B,T,_=emb.shape; h=self.input_proj(emb)+self.pos_enc[:,:T,:]
        if site_ids is not None and hours is not None:
            meta=self.meta_proj(torch.cat([self.site_emb(site_ids),self.hour_emb(hours)],dim=-1))
            h=h+meta[:,None,:]
        for i,(fwd,bwd,merge,norm) in enumerate(zip(self.ssm_fwd,self.ssm_bwd,self.ssm_merge,self.ssm_norm)):
            res=h; hf=fwd(h); hb=bwd(h.flip(1)).flip(1)
            h=self.drop(merge(torch.cat([hf,hb],dim=-1))); h=norm(h+res)
            if self.use_cross_attn:
                attn_out,_=self.cross_attn[i](h,h,h); h=self.cross_norm[i](h+attn_out)
        h_n=F.normalize(h,dim=-1); p_n=F.normalize(self.prototypes,dim=-1)
        sim=torch.matmul(h_n,p_n.T)*F.softplus(self.proto_temp)+self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha=torch.sigmoid(self.fusion_alpha)[None,None,:]
            out=alpha*sim+(1-alpha)*perch_logits
        else: out=sim
        return out

class ResidualSSM(nn.Module):
    def __init__(self,d_input=1536,d_scores=234,d_model=64,d_state=8,n_classes=234,
                 n_windows=12,dropout=0.1,n_sites=20,meta_dim=8):
        super().__init__(); self.n_classes=n_classes
        self.input_proj=nn.Sequential(nn.Linear(d_input+d_scores,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.ssm_fwd=SelectiveSSM(d_model,d_state); self.ssm_bwd=SelectiveSSM(d_model,d_state)
        self.ssm_merge=nn.Linear(2*d_model,d_model); self.ssm_norm=nn.LayerNorm(d_model); self.ssm_drop=nn.Dropout(dropout)
        self.output_head=nn.Linear(d_model,n_classes)
        nn.init.zeros_(self.output_head.weight); nn.init.zeros_(self.output_head.bias)
    def forward(self,emb,first_pass,site_ids=None,hours=None):
        B,T,_=emb.shape; x=torch.cat([emb,first_pass],dim=-1)
        h=self.input_proj(x)+self.pos_enc[:,:T,:]
        if site_ids is not None and hours is not None:
            meta=self.meta_proj(torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                                            self.hour_emb(hours.clamp(0,23))],dim=-1))
            h=h+meta.unsqueeze(1)
        res=h; hf=self.ssm_fwd(h); hb=self.ssm_bwd(h.flip(1)).flip(1)
        h=self.ssm_drop(self.ssm_merge(torch.cat([hf,hb],dim=-1))); h=self.ssm_norm(h+res)
        return self.output_head(h)

def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=40, patience=8, lr=1e-3, n_sites=20, verbose=False):
    n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
    log_f=scores_full.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    fnames=meta_full["filename"].unique(); sites_u=sorted(meta_full["site"].unique())
    site2i={s:i+1 for i,s in enumerate(sites_u)}
    site_ids=np.array([min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0],0),n_sites-1) for fn in fnames],dtype=np.int64)
    hour_ids=np.array([int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0])%24 for fn in fnames],dtype=np.int64)
    model=LightProtoSSM(n_classes=N_CLASSES,n_sites=n_sites,use_cross_attn=True,cross_attn_heads=2)
    model.init_prototypes(torch.tensor(emb_full,dtype=torch.float32),torch.tensor(Y_full,dtype=torch.float32))
    emb_t=torch.tensor(emb_f,dtype=torch.float32); log_t=torch.tensor(log_f,dtype=torch.float32)
    lab_t=torch.tensor(lab_f,dtype=torch.float32); site_t=torch.tensor(site_ids,dtype=torch.long)
    hour_t=torch.tensor(hour_ids,dtype=torch.long)
    pos_cnt=lab_t.sum(dim=(0,1)); total=lab_t.shape[0]*lab_t.shape[1]
    pos_weight=((total-pos_cnt)/(pos_cnt+1)).clamp(max=25.0)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
    best_loss,best_state,wait=float("inf"),None,0
    swa_model=torch.optim.swa_utils.AveragedModel(model); swa_start=int(n_epochs*0.65)
    swa_sched=torch.optim.swa_utils.SWALR(opt,swa_lr=4e-4)
    for ep in range(n_epochs):
        model.train()
        out=model(emb_t,log_t,site_ids=site_t,hours=hour_t)
        loss=F.binary_cross_entropy_with_logits(out,lab_t,pos_weight=pos_weight[None,None,:])+0.15*F.mse_loss(out,log_t)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if ep>=swa_start: swa_model.update_parameters(model); swa_sched.step()
        else: sched.step()
        if loss.item()<best_loss:
            best_loss=loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
    if ep>=swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0),swa_model); model=swa_model
    else: model.load_state_dict(best_state)
    model.eval(); return model,site2i

def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=[0,1,-1,2,-2]):
    proto_model.eval(); all_preds=[]
    emb_t=torch.tensor(emb_files,dtype=torch.float32); sc_t=torch.tensor(sc_files,dtype=torch.float32)
    for shift in shifts:
        e=torch.roll(emb_t,shift,dims=1) if shift else emb_t
        s=torch.roll(sc_t,shift,dims=1) if shift else sc_t
        with torch.no_grad():
            out=proto_model(e,s,site_ids=site_t,hours=hour_t).numpy()
        if shift: out=np.roll(out,-shift,axis=1)
        all_preds.append(out)
    return np.mean(all_preds,axis=0)

def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids,
                        n_epochs=30, patience=8, lr=1e-3, correction_weight=0.30, verbose=False):
    n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
    fp_f=first_pass_flat.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    fp_prob=1.0/(1.0+np.exp(-np.clip(fp_f,-30,30))); residuals=lab_f-fp_prob
    n_val=max(1,int(n_files*0.15)); rng=torch.Generator(); rng.manual_seed(42)
    perm=torch.randperm(n_files,generator=rng).numpy(); val_i=perm[:n_val]; train_i=perm[n_val:]
    emb_t=torch.tensor(emb_f,dtype=torch.float32); fp_t=torch.tensor(fp_f,dtype=torch.float32)
    res_t=torch.tensor(residuals,dtype=torch.float32)
    site_t=torch.tensor(site_ids,dtype=torch.long); hour_t=torch.tensor(hour_ids,dtype=torch.long)
    model=ResidualSSM(n_classes=N_CLASSES)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
    best_loss,best_state,wait=float("inf"),None,0
    for ep in range(n_epochs):
        model.train()
        corr=model(emb_t[train_i],fp_t[train_i],site_ids=site_t[train_i],hours=hour_t[train_i])
        loss=F.mse_loss(corr,res_t[train_i])
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            val_corr=model(emb_t[val_i],fp_t[val_i],site_ids=site_t[val_i],hours=hour_t[val_i])
            val_loss=F.mse_loss(val_corr,res_t[val_i])
        if val_loss.item()<best_loss:
            best_loss=val_loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(best_state); return model,correction_weight

print("Sequence Models defined")


---
## Cell 11 — Test inference

Finds hidden test soundscapes. If hidden test files are absent, it runs a dry-run on a small number of training files for notebook validation.

In [ ]:
# ── Test inference ────────────────────────────────────────────────────────────
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")

meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"Test scores: {sc_te.shape}")


---
## Cell 12 — Full ProtoSSM pipeline

Trains ProtoSSM, applies priors and MLP probes, calibrates thresholds, trains ResidualSSM, then saves submission_protossm.csv.

In [ ]:
# ── Full ProtoSSM pipeline ────────────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=40, patience=8, lr=1e-3, verbose=False)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

n_test_files  = len(sc_te) // N_WINDOWS
emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
n_sites_cap   = 20
test_site_ids = np.array([min(site2i_tr.get(meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                           for fn in test_fnames], dtype=np.int64)
test_hour_ids = np.array([int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0])%24
                           for fn in test_fnames], dtype=np.int64)

proto_model.eval()
with torch.no_grad():
    proto_out = proto_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(sc_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

prior_tables   = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(sc_te, sites=meta_te["site"].to_numpy(),
                              hours=meta_te["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=0.4)

probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned, min_pos=5, pca_dim=64, alpha_blend=0.4)
sc_te_adjusted = apply_mlp_probes_vectorized(emb_te, sc_te_adjusted, probe_models, emb_scaler, emb_pca, alpha_blend)

ENSEMBLE_W      = 0.5
first_pass_flat = (ENSEMBLE_W * proto_scores_flat + (1.0 - ENSEMBLE_W) * sc_te_adjusted)

n_tr_files    = len(sc_tr) // N_WINDOWS
emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
tr_site_ids   = np.array([min(site2i_tr.get(meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                           for fn in tr_fnames], dtype=np.int64)
tr_hour_ids   = np.array([int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0])%24
                           for fn in tr_fnames], dtype=np.int64)

proto_tr_out = run_tta_proto(proto_model, emb_tr_f, sc_tr_f,
    site_t=torch.tensor(tr_site_ids, dtype=torch.long),
    hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2])
proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

sc_tr_prior = apply_prior(sc_tr, sites=meta_tr["site"].to_numpy(),
                           hours=meta_tr["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=0.4)
sc_tr_mlp = apply_mlp_probes_vectorized(emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
first_pass_tr = (ENSEMBLE_W * proto_tr_flat + (1.0 - ENSEMBLE_W) * sc_tr_mlp)

train_probs_for_calib = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_for_calib, Y_FULL=Y_FULL_aligned,
    threshold_grid=[0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70], n_windows=N_WINDOWS)

t0 = time.time()
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr, first_pass_flat=first_pass_tr, Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids, hour_ids=tr_hour_ids, n_epochs=30, patience=8, lr=1e-3,
    correction_weight=0.30, verbose=False)
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f,         dtype=torch.float32),
        torch.tensor(first_pass_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = first_pass_flat + correction_weight * correction_flat
final_scores    = final_scores / temperatures[None, :]
probs = sigmoid(final_scores)
probs = file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.4)
probs = rank_aware_scaling(probs,    n_windows=N_WINDOWS, power=0.4)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20)
probs = np.clip(probs, 0.0, 1.0)
probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)   # ← now applied

sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub.insert(0, "row_id", meta_te["row_id"].values)
sub.to_csv("submission_protossm.csv", index=False)
print("ProtoSSM execution complete")
print(f"Total wall time so far: {(time.time() - _WALL_START)/60:.1f} min")
del emb_tr_f, sc_tr_f, proto_model, res_model
gc.collect()
print("Memory freed. Ready for SED cell.")


---
## Cell 13 — Distilled SED branch

This branch converts each 5-second window into a mel spectrogram and runs the public distilled SED ONNX fold ensemble. It is independent from ProtoSSM, which makes it useful for rank blending because it may catch short calls that the Perch/SSM branch misses.

```text
60s audio → 12 × 5s chunks → mel spectrogram → SED folds → clip/frame blend → temporal smoothing → submission_sed.csv
```

In [ ]:
import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80

def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    n = 60 * SR
    if len(y) < n: y = np.pad(y, (0, n - len(y)))
    else:          y = y[:n]
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    return chunks, ends

def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

# Use the same test files as Cell 1
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    dry_n = CFG["dryrun_n_files"] if "CFG" in dir() else 20
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:(dry_n or 20)]

sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                         key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")

sed_rows, sed_preds = [], []

for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel = audio_to_mel(chunks)
    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)

    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]             # (12, 234)
        frame_max   = outs[1].max(axis=1) # (12, 234)
        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

    p_mean = p_sum / len(sed_sessions)

    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)

    stem = path.stem
    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)

    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)}")

sed_preds_arr = np.concatenate(sed_preds, axis=0)
sed_sub = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
sed_sub.insert(0, "row_id", sed_rows)
sed_sub.to_csv("submission_sed.csv", index=False)
print(f"Distilled SED Processing Complete. Shape: {sed_sub.shape}")


---
## Cell 14 — Optional BirdNET branch

BirdNET uses 3-second chunks, so the notebook maps those chunks back into the competition's 5-second windows using overlap-based max pooling. If the BirdNET model is missing, this cell creates a zero-filled fallback so the final blend cell still runs safely.

```text
60s audio → 20 × 3s BirdNET chunks → species probabilities → overlap map → 12 × 5s rows → submission_birdnet.csv
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BirdNET v2.4 — Third model branch (NEW)
# Uses: shadiakiki1/birdnet-analyzer/liteRT/birdnet_global_6k_v2.4_model_fp32-1
# This cell is safe to run even if BirdNET model is not attached —
# it will gracefully set USE_BIRDNET=False and skip.
# ══════════════════════════════════════════════════════════════════════════════
import librosa as _librosa
from scipy.ndimage import gaussian_filter1d as _gf1d

BIRDNET_SR = 48_000
BIRDNET_CHUNK_SEC = 3
BIRDNET_CHUNK_SAMPLES = BIRDNET_SR * BIRDNET_CHUNK_SEC  # 144000

def _find_birdnet_model():
    for pat in ["**/birdnet_global_6k_v2.4_model_fp32*.tflite",
                "**/BirdNET_GLOBAL_6K_V2.4_Model_FP32*.tflite",
                "**/*birdnet*fp32*.tflite",
                "**/*birdnet*.tflite"]:
        hits = sorted(Path("/kaggle/input").rglob(pat))
        if hits:
            print(f"  Found BirdNET model: {hits[0].name}")
            return hits[0]
    return None

def _find_birdnet_labels():
    for pat in ["**/BirdNET_GLOBAL_6K_V2.4_Labels.txt",
                "**/birdnet*labels*.txt",
                "**/*birdnet*label*.txt"]:
        hits = sorted(Path("/kaggle/input").rglob(pat))
        if hits:
            print(f"  Found BirdNET labels: {hits[0].name}")
            return hits[0]
    return None

_bn_model_path  = _find_birdnet_model()
_bn_labels_path = _find_birdnet_labels()

if _bn_model_path is None:
    USE_BIRDNET = False
    print("BirdNET model not found — will use original 60/40 blend")
    BN_TO_COMP = {}
    BN_PROXY   = {}
else:
    USE_BIRDNET = True
    try:
        from tflite_runtime.interpreter import Interpreter as _TFLiteInterp
    except ImportError:
        from tensorflow.lite.python.interpreter import Interpreter as _TFLiteInterp

    _bn_interp = _TFLiteInterp(model_path=str(_bn_model_path), num_threads=4)
    _bn_interp.allocate_tensors()
    _bn_in  = _bn_interp.get_input_details()[0]
    _bn_out = _bn_interp.get_output_details()
    _bn_logit_idx = _bn_out[-1]["index"]
    print(f"  BirdNET input:  {_bn_in['shape']}")
    print(f"  BirdNET output: {_bn_out[-1]['shape']}")

    # Load labels
    if _bn_labels_path:
        _bn_labels_raw = [l.strip() for l in _bn_labels_path.read_text().splitlines() if l.strip()]
    else:
        _bn_labels_raw = []
        print("  WARNING: BirdNET labels not found — species mapping will be empty")

    # Parse scientific names (before first underscore)
    _bn_sci = [lbl.split("_", 1)[0].strip() for lbl in _bn_labels_raw]

    # Direct mapping: BirdNET index → competition class index via scientific name
    _tax_sci = taxonomy.set_index("scientific_name")["primary_label"].to_dict()
    BN_TO_COMP = {}
    for bn_i, sci in enumerate(_bn_sci):
        if sci in _tax_sci and _tax_sci[sci] in label_to_idx:
            BN_TO_COMP[bn_i] = label_to_idx[_tax_sci[sci]]

    # Genus-level proxy for unmapped competition classes
    _mapped_comp = set(BN_TO_COMP.values())
    BN_PROXY = {}  # comp_class_idx → list of bn_indices
    for ci, primary in enumerate(PRIMARY_LABELS):
        if ci in _mapped_comp:
            continue
        row = taxonomy[taxonomy["primary_label"] == primary]
        if row.empty:
            continue
        genus = str(row.iloc[0]["scientific_name"]).split()[0]
        idxs = [i for i, s in enumerate(_bn_sci) if s.startswith(genus + " ")]
        if idxs:
            BN_PROXY[ci] = idxs

    n_mapped = len(set(BN_TO_COMP.values()) | set(BN_PROXY.keys()))
    print(f"  BirdNET→competition: {len(BN_TO_COMP)} direct + {len(BN_PROXY)} genus-proxy = {n_mapped}/{N_CLASSES} classes")

# Window overlap: each 5s competition window overlaps certain 3s BirdNET chunks
_N_BN_CHUNKS = 20  # 60s / 3s
_win_to_chunks = []
for w in range(N_WINDOWS):
    ws, we = w * 5, (w + 1) * 5
    _win_to_chunks.append([j for j in range(_N_BN_CHUNKS)
                            if 3*j < we and 3*(j+1) > ws])

def run_birdnet(paths, verbose=True):
    if not USE_BIRDNET or not _bn_labels_raw:
        return None, None

    paths = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    wr = 0
    itr = tqdm(paths, desc="BirdNET") if verbose else paths

    for path in itr:
        y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1)
        if sr0 != BIRDNET_SR:
            y = _librosa.resample(y, orig_sr=sr0, target_sr=BIRDNET_SR)
        tgt = 60 * BIRDNET_SR
        if len(y) < tgt: y = np.pad(y, (0, tgt - len(y)))
        else:             y = y[:tgt]

        chunks = y.reshape(_N_BN_CHUNKS, BIRDNET_CHUNK_SAMPLES)
        chunk_probs = np.zeros((_N_BN_CHUNKS, len(_bn_labels_raw)), dtype=np.float32)
        for j, chunk in enumerate(chunks):
            _bn_interp.set_tensor(_bn_in["index"], chunk[None, :].astype(np.float32))
            _bn_interp.invoke()
            logits = _bn_interp.get_tensor(_bn_logit_idx)[0]
            chunk_probs[j] = 1.0 / (1.0 + np.exp(-np.clip(logits, -50, 50)))

        # Aggregate 3s chunks → 5s competition windows via max pooling
        stem = path.stem
        for w, clist in enumerate(_win_to_chunks):
            wp = chunk_probs[clist].max(axis=0)
            r = wr + w
            row_ids[r]   = f"{stem}_{(w+1)*5}"
            filenames[r] = path.name
            for bn_i, ci in BN_TO_COMP.items():
                if wp[bn_i] > scores[r, ci]:
                    scores[r, ci] = wp[bn_i]
            for ci, bn_idxs in BN_PROXY.items():
                v = wp[bn_idxs].max()
                if v > scores[r, ci]:
                    scores[r, ci] = v
        wr += N_WINDOWS

    meta_df = pd.DataFrame({"row_id": row_ids[:wr], "filename": filenames[:wr]})
    return meta_df, scores[:wr]

print("BirdNET inference engine defined")

# Run BirdNET on test files
_test_paths_bn = sorted((BASE / "test_soundscapes").glob("*.ogg"))
if len(_test_paths_bn) == 0:
    _dry_n = 20
    _test_paths_bn = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:_dry_n]

if USE_BIRDNET:
    t0 = time.time()
    _meta_bn, _scores_bn = run_birdnet(_test_paths_bn, verbose=True)
    print(f"BirdNET inference: {time.time()-t0:.1f}s  shape={_scores_bn.shape}")

    # Smooth BirdNET predictions (same sigma as SED)
    _scores_bn_v = _scores_bn.reshape(len(_scores_bn)//N_WINDOWS, N_WINDOWS, N_CLASSES)
    for fi in range(len(_scores_bn_v)):
        _scores_bn_v[fi] = _gf1d(_scores_bn_v[fi], sigma=0.65, axis=0, mode="nearest")
    _scores_bn = _scores_bn_v.reshape(-1, N_CLASSES)

    _bn_sub = pd.DataFrame(np.clip(_scores_bn, 0.0, 1.0), columns=PRIMARY_LABELS)
    _bn_sub.insert(0, "row_id", _meta_bn["row_id"].values)
    _bn_sub.to_csv("submission_birdnet.csv", index=False)
    covered = (_scores_bn > 0.01).any(axis=0).sum()
    print(f"BirdNET saved. Coverage: {covered}/{N_CLASSES} classes")
else:
    # Save zero-filled CSV so blend cell always has the file
    _dummy = pd.read_csv("submission_protossm.csv")
    for c in PRIMARY_LABELS: _dummy[c] = 0.0
    _dummy.to_csv("submission_birdnet.csv", index=False)
    print("BirdNET unavailable — zero submission saved")


---
## Cell 15A — A2' mel-spectrum branch (EfficientNetV2-S)

This is the first real A2' experiment: add a **true mel-spectrum classifier** branch using Baiyu's public 4-fold EfficientNetV2-S checkpoints.

Why this branch matters:
- it does not read Perch embeddings
- it is trained in spectrogram space
- it gives us a cleaner test of whether a genuine second acoustic family can move us beyond the public 0.946 ceiling

This notebook writes:
- `submission_effv2s.csv`
- `effv2s_branch_summary.csv`

The final blend cell then emits multiple candidate CSVs so we can compare a conservative and a stronger A2' mix without rewriting the whole pipeline again.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio

DEFAULT_TOP_DB = 80.0


def find_baiyu_dir(input_root: Path) -> Path | None:
    hits = sorted(input_root.rglob("fold0_best.pth"))
    return hits[0].parent if hits else None


class BaiyuEffV2S(nn.Module):
    """
    Best-effort reconstruction of Baiyu's EfficientNetV2-S distillation branch.

    The checkpoint structure is:
      - bb.*       : EfficientNetV2-S backbone
      - att.0/2    : attention MLP over time frames
      - fc_att     : framewise classifier for attention-pooled logits
      - fc_max     : framewise classifier for max-pooled logits

    This keeps the architecture simple and close to the saved state dict layout,
    which makes it easy to audit and iterate on.
    """

    def __init__(self, n_classes: int, backbone_name: str = "tf_efficientnetv2_s.in21k_ft_in1k"):
        super().__init__()
        import timm

        self.bb = timm.create_model(
            backbone_name,
            pretrained=False,
            in_chans=1,
            num_classes=0,
            global_pool="",
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.att = nn.Sequential(
            nn.Linear(1280, 640),
            nn.ReLU(inplace=True),
            nn.Linear(640, n_classes),
        )
        self.fc_att = nn.Linear(1280, n_classes)
        self.fc_max = nn.Linear(1280, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.bb(x)
        if isinstance(f, (tuple, list)):
            f = f[-1]
        f = self.freq_pool(f).squeeze(2).permute(0, 2, 1)
        att = torch.softmax(self.att(f), dim=1)
        logits_att = self.fc_att(f)
        logits_max = self.fc_max(f)
        pooled_att = (att * logits_att).sum(dim=1)
        pooled_max = logits_max.max(dim=1).values
        return 0.5 * (pooled_att + pooled_max)


def load_baiyu_models(
    ckpt_dir: Path,
    n_classes: int,
    device: torch.device,
    max_folds: int | None = None,
) -> list[nn.Module]:
    import timm  # noqa: F401  # trigger ImportError early if missing

    ckpts = sorted(ckpt_dir.glob("fold*_best.pth"))
    if max_folds is not None:
        ckpts = ckpts[:max_folds]
    models: list[nn.Module] = []
    for ckpt_path in ckpts:
        # Public checkpoint contains a small amount of non-tensor metadata.
        # PyTorch 2.6 defaults to weights_only=True, which rejects that payload.
        payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
        model = BaiyuEffV2S(n_classes=n_classes)
        model.load_state_dict(state, strict=True)
        model.to(device).eval()
        models.append(model)
    return models


def build_effv2s_transforms(
    sr: int,
    *,
    n_fft: int = 2048,
    hop_length: int = 512,
    n_mels: int = 128,
    f_min: int = 20,
    f_max: int = 16000,
    top_db: float = DEFAULT_TOP_DB,
):
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        f_min=f_min,
        f_max=f_max,
        power=2.0,
    )
    to_db = torchaudio.transforms.AmplitudeToDB(top_db=top_db)
    return mel, to_db


def file_to_chunks(path: Path, sr: int, file_samples: int, n_windows: int, window_samples: int) -> tuple[torch.Tensor, np.ndarray]:
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    x = torch.from_numpy(y)
    if sr0 != sr:
        x = torchaudio.functional.resample(x, sr0, sr)
    if x.numel() < file_samples:
        x = torch.nn.functional.pad(x, (0, file_samples - x.numel()))
    else:
        x = x[:file_samples]
    chunks = x.view(n_windows, window_samples)
    ends = np.arange(1, n_windows + 1) * (window_samples // sr)
    return chunks, ends


def chunks_to_effv2s_input(
    chunks: torch.Tensor,
    mel,
    to_db,
    *,
    norm_mode: str = "zscore",
    top_db: float = DEFAULT_TOP_DB,
) -> torch.Tensor:
    specs = mel(chunks)
    specs = to_db(specs)
    if norm_mode == "zscore":
        means = specs.mean(dim=(1, 2), keepdim=True)
        stds = specs.std(dim=(1, 2), keepdim=True).clamp_min(1e-6)
        specs = (specs - means) / stds
    elif norm_mode == "db01":
        specs = ((specs + top_db) / top_db).clamp(0.0, 1.0)
    else:
        raise ValueError(f"Unknown norm_mode={norm_mode}")
    return specs.unsqueeze(1)


def default_effv2s_variants(for_public_dryrun: bool) -> list[dict[str, object]]:
    base = {
        "name": "base_128_h512_z",
        "n_fft": 2048,
        "hop_length": 512,
        "n_mels": 128,
        "f_min": 20,
        "f_max": 16000,
        "top_db": DEFAULT_TOP_DB,
        "norm_mode": "zscore",
    }
    if not for_public_dryrun:
        return [base]
    return [
        base,
        {
            "name": "hop320_128_z",
            "n_fft": 2048,
            "hop_length": 320,
            "n_mels": 128,
            "f_min": 20,
            "f_max": 16000,
            "top_db": DEFAULT_TOP_DB,
            "norm_mode": "zscore",
        },
        {
            "name": "mel256_h320_z",
            "n_fft": 2048,
            "hop_length": 320,
            "n_mels": 256,
            "f_min": 20,
            "f_max": 16000,
            "top_db": DEFAULT_TOP_DB,
            "norm_mode": "zscore",
        },
    ]


def build_file_label_map(soundscape_labels: pd.DataFrame, filenames: Iterable[str]) -> dict[str, set[str]]:
    wanted = set(filenames)
    label_map: dict[str, set[str]] = {}
    subset = soundscape_labels[soundscape_labels["filename"].isin(wanted)]
    for filename, group in subset.groupby("filename"):
        labels: set[str] = set()
        for raw in group["primary_label"].dropna().astype(str):
            for token in raw.split(";"):
                token = token.strip()
                if token:
                    labels.add(token)
        label_map[Path(filename).stem] = labels
    return label_map


def evaluate_file_level_topk(
    pred_df: pd.DataFrame,
    *,
    file_label_map: dict[str, set[str]],
    topk: int = 5,
) -> pd.DataFrame:
    if pred_df.empty or not file_label_map:
        return pd.DataFrame(columns=["file_stem", "topk", "labels", "hit_topk"])

    pred_df = pred_df.copy()
    pred_df["file_stem"] = pred_df["row_id"].astype(str).str.rsplit("_", n=1).str[0]
    prob_cols = [c for c in pred_df.columns if c not in {"row_id", "file_stem"}]

    rows: list[dict[str, object]] = []
    for file_stem, group in pred_df.groupby("file_stem"):
        labels = sorted(file_label_map.get(file_stem, set()))
        if not labels:
            continue
        class_means = group[prob_cols].mean(axis=0).sort_values(ascending=False)
        top_labels = class_means.head(topk).index.tolist()
        rows.append(
            {
                "file_stem": file_stem,
                "topk": ";".join(top_labels),
                "labels": ";".join(labels),
                "hit_topk": bool(set(top_labels) & set(labels)),
            }
        )
    return pd.DataFrame(rows)


@torch.inference_mode()
def run_baiyu_branch(
    test_paths: Iterable[Path],
    *,
    base_dir: Path,
    input_root: Path,
    primary_labels: list[str],
    sr: int,
    file_samples: int,
    n_windows: int,
    window_samples: int,
    soundscape_labels_df: pd.DataFrame | None = None,
    batch_size: int = 12,
    max_folds: int | None = None,
    variants: list[dict[str, object]] | None = None,
) -> dict[str, object]:
    result = {
        "active": False,
        "reason": "",
        "n_models": 0,
        "device": "cpu",
        "rows": 0,
        "output_csv": "submission_effv2s.csv",
        "summary_csv": "effv2s_branch_summary.csv",
        "selected_variant": "",
    }

    test_paths = [Path(p) for p in test_paths]

    ckpt_dir = find_baiyu_dir(input_root)
    if ckpt_dir is None:
        result["reason"] = "baiyuby checkpoints not found"
        return result

    try:
        import timm  # noqa: F401
    except ImportError:
        result["reason"] = "timm unavailable"
        return result

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = load_baiyu_models(ckpt_dir, len(primary_labels), device, max_folds=max_folds)
    if not models:
        result["reason"] = "no Baiyu checkpoints loaded"
        return result

    for_public_dryrun = not any("Test" in p.name for p in test_paths)
    variants = variants or default_effv2s_variants(for_public_dryrun=for_public_dryrun)
    file_label_map = (
        build_file_label_map(soundscape_labels_df, [p.name for p in test_paths])
        if soundscape_labels_df is not None else {}
    )

    variant_outputs: dict[str, pd.DataFrame] = {}
    summary_rows: list[dict[str, object]] = []
    sanity_outputs: list[pd.DataFrame] = []

    for variant in variants:
        mel, to_db = build_effv2s_transforms(
            sr,
            n_fft=int(variant["n_fft"]),
            hop_length=int(variant["hop_length"]),
            n_mels=int(variant["n_mels"]),
            f_min=int(variant["f_min"]),
            f_max=int(variant["f_max"]),
            top_db=float(variant["top_db"]),
        )
        mel = mel.to(device)
        to_db = to_db.to(device)

        row_ids: list[str] = []
        preds: list[np.ndarray] = []

        for path in test_paths:
            chunks, ends = file_to_chunks(path, sr, file_samples, n_windows, window_samples)
            x = chunks_to_effv2s_input(
                chunks,
                mel,
                to_db,
                norm_mode=str(variant["norm_mode"]),
                top_db=float(variant["top_db"]),
            ).to(device)

            fold_sum = torch.zeros((x.shape[0], len(primary_labels)), device=device)
            for model in models:
                for start in range(0, x.shape[0], batch_size):
                    xb = x[start:start + batch_size]
                    fold_sum[start:start + batch_size] += torch.sigmoid(model(xb))
            probs = (fold_sum / len(models)).cpu().numpy().astype(np.float32)

            row_ids.extend([f"{path.stem}_{int(t)}" for t in ends])
            preds.append(probs)

        pred_arr = np.concatenate(preds, axis=0) if preds else np.zeros((0, len(primary_labels)), dtype=np.float32)
        sub = pd.DataFrame(np.clip(pred_arr, 0.0, 1.0), columns=primary_labels)
        sub.insert(0, "row_id", row_ids)
        variant_name = str(variant["name"])
        sub.to_csv(base_dir / f"submission_effv2s_{variant_name}.csv", index=False)
        variant_outputs[variant_name] = sub

        sanity_df = evaluate_file_level_topk(sub, file_label_map=file_label_map, topk=5)
        if not sanity_df.empty:
            sanity_df = sanity_df.assign(variant=variant_name)
            sanity_outputs.append(sanity_df)

        hit_rate = float(sanity_df["hit_topk"].mean()) if not sanity_df.empty else np.nan
        summary_rows.append(
            {
                "branch": "baiyu_effv2s",
                "variant": variant_name,
                "active": True,
                "checkpoint_dir": str(ckpt_dir),
                "n_models": len(models),
                "device": str(device),
                "rows": len(sub),
                "n_fft": int(variant["n_fft"]),
                "hop_length": int(variant["hop_length"]),
                "n_mels": int(variant["n_mels"]),
                "norm_mode": str(variant["norm_mode"]),
                "mean_prob": float(pred_arr.mean()) if pred_arr.size else 0.0,
                "max_prob": float(pred_arr.max()) if pred_arr.size else 0.0,
                "sanity_files": int(len(sanity_df)),
                "sanity_hits_top5": int(sanity_df["hit_topk"].sum()) if not sanity_df.empty else 0,
                "sanity_hit_rate": hit_rate,
            }
        )

    summary = pd.DataFrame(summary_rows)
    if not summary.empty:
        sort_key = summary["sanity_hit_rate"].fillna(-1.0) * 1000.0 + summary["mean_prob"]
        selected_idx = int(sort_key.to_numpy().argmax())
        selected_variant = str(summary.iloc[selected_idx]["variant"])
        summary["selected"] = summary["variant"].eq(selected_variant)
    else:
        selected_variant = ""
        summary["selected"] = False

    summary.to_csv(base_dir / "effv2s_branch_summary.csv", index=False)
    if sanity_outputs:
        pd.concat(sanity_outputs, ignore_index=True).to_csv(base_dir / "effv2s_sanity_file_summary.csv", index=False)

    if selected_variant:
        variant_outputs[selected_variant].to_csv(base_dir / "submission_effv2s.csv", index=False)
    else:
        pd.DataFrame(columns=["row_id", *primary_labels]).to_csv(base_dir / "submission_effv2s.csv", index=False)

    result.update({
        "active": True,
        "reason": "",
        "n_models": len(models),
        "device": str(device),
        "rows": int(summary["rows"].max()) if not summary.empty else 0,
        "selected_variant": selected_variant,
    })
    return result

from pathlib import Path

TEST_PATHS_EFF = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_PUBLIC_DRYRUN = len(TEST_PATHS_EFF) == 0
if IS_PUBLIC_DRYRUN:
    TEST_PATHS_EFF = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:20]

EFFV2S_INFO = run_baiyu_branch(
    TEST_PATHS_EFF,
    base_dir=Path('.'),
    input_root=INPUT_ROOT,
    primary_labels=PRIMARY_LABELS,
    sr=SR,
    file_samples=FILE_SAMPLES,
    n_windows=N_WINDOWS,
    window_samples=WINDOW_SAMPLES,
    soundscape_labels_df=soundscape_labels,
    batch_size=12,
    max_folds=None,
    variants=None,
)

print("EffV2-S branch active:", EFFV2S_INFO['active'])
print("EffV2-S branch reason:", EFFV2S_INFO['reason'])
print("EffV2-S branch models:", EFFV2S_INFO['n_models'])
print("EffV2-S branch rows:", EFFV2S_INFO['rows'])
print("[A2'] selected variant =", EFFV2S_INFO.get('selected_variant', ''))

try:
    branch_summary = pd.read_csv('effv2s_branch_summary.csv')
    show_cols = [c for c in [
        'variant', 'n_mels', 'hop_length', 'norm_mode',
        'sanity_hit_rate', 'sanity_hits_top5', 'sanity_files',
        'mean_prob', 'max_prob', 'selected'
    ] if c in branch_summary.columns]
    print("[A2'] branch variant summary:")
    print(branch_summary[show_cols].to_string(index=False))
except Exception as _a2_branch_e:
    print("[A2'] branch summary failed:", _a2_branch_e)

try:
    sanity_path = Path('effv2s_sanity_file_summary.csv')
    if sanity_path.exists():
        sanity_df = pd.read_csv(sanity_path)
        selected_variant = EFFV2S_INFO.get('selected_variant', '')
        if selected_variant:
            sel = sanity_df[sanity_df['variant'] == selected_variant].copy()
            if not sel.empty:
                print(f"[A2'] sanity hit rate ({selected_variant}) = {sel['hit_topk'].mean():.3f}")
                print("[A2'] sanity sample rows:")
                print(sel[['file_stem', 'labels', 'topk', 'hit_topk']].head(3).to_string(index=False))
except Exception as _a2_sanity_e:
    print("[A2'] sanity failed:", _a2_sanity_e)


---
## Cell 15 — Final rank blend and post-processing gates

The final step uses rank-based blending rather than raw probability averaging. This reduces calibration mismatch between ProtoSSM, SED, and BirdNET.

| Gate | Purpose | Intuition |
|---|---|---|
| Gate 1 | Noise suppression | If SED strongly disagrees with a high ProtoSSM score, slightly trust ProtoSSM rank |
| Gate 2 | Temporal continuity | Protect continuous calls across neighboring windows |
| Gate 3 | SED spike preservation | Rescue brief high-confidence SED events |
| Gate 3b | BirdNET spike preservation | Rescue brief BirdNET-only signals when available |
| Gate 4 | Sonotype mirroring | Share scores across visually/sound-alike sonotype groups |
| Gate 5 | Rare-class thresholding | Slightly suppress weak rare-class noise |

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

PROTOSSM_CSV = "submission_protossm.csv"
SED_CSV      = "submission_sed.csv"
BIRDNET_CSV  = "submission_birdnet.csv"
EFFV2S_CSV   = "submission_effv2s.csv"
OUT_CSV      = "submission.csv"
EPS = 1e-5

df_proto = pd.read_csv(PROTOSSM_CSV)
df_sed   = pd.read_csv(SED_CSV)
cols = [c for c in df_proto.columns if c != "row_id"]

df_sed = df_sed.set_index("row_id").loc[df_proto["row_id"]].reset_index()
p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
p_sed   = np.clip(df_sed[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
rank_proto = pd.DataFrame(p_proto).rank(axis=0, pct=True).to_numpy(np.float32)
rank_sed   = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)

rank_birdnet = None
p_birdnet = None
if os.path.exists(BIRDNET_CSV):
    df_bn = pd.read_csv(BIRDNET_CSV)
    df_bn = df_bn.set_index("row_id").loc[df_proto["row_id"]].reset_index()
    p_birdnet = np.clip(df_bn[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
    if (p_birdnet > 0.01).any():
        rank_birdnet = pd.DataFrame(p_birdnet).rank(axis=0, pct=True).to_numpy(np.float32)

rank_effv2s = None
p_effv2s = None
if os.path.exists(EFFV2S_CSV):
    df_eff = pd.read_csv(EFFV2S_CSV)
    df_eff = df_eff.set_index("row_id").loc[df_proto["row_id"]].reset_index()
    p_effv2s = np.clip(df_eff[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
    if (p_effv2s > 0.01).any():
        rank_effv2s = pd.DataFrame(p_effv2s).rank(axis=0, pct=True).to_numpy(np.float32)

row_ids = df_proto["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

# Shared gates from the 0.946 baseline.
def apply_postprocess(pred):
    fake_only = (p_proto > 0.50) & (p_sed < 0.05)
    pred = np.where(fake_only, (1.0 - 0.08) * pred + 0.08 * rank_proto, pred)

    offs = np.arange(-3, 4, dtype=np.float32)
    proto_kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
    proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)

    pa_ctx = p_proto.copy()
    for fid in pd.unique(file_ids):
        m = file_ids == fid
        x = p_proto[m]
        if len(x) > 1:
            xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
            pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(7))

    xctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
    proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only)
    pred = np.where(proto_cont, (1.0 - 0.15) * pred + 0.15 * np.maximum(rank_proto, xctx), pred)

    sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
    pred = np.where(sed_only, (1.0 - 0.12) * pred + 0.12 * rank_sed, pred)

    if rank_birdnet is not None:
        bn_only = (rank_birdnet > 0.95) & (rank_proto < 0.75) & (rank_sed < 0.80) & (~fake_only) & (~proto_cont) & (~sed_only)
        pred = np.where(bn_only, (1.0 - 0.10) * pred + 0.10 * rank_birdnet, pred)

    out = df_proto.copy()
    out[cols] = pred.astype(np.float32)

    mirror_pairs = (
        ("47158son15", "47158son16"),
        ("47158son09", "47158son12"),
        ("47158son02", "47158son14"),
        ("47158son13", "47158son21", "47158son22", "47158son23"),
    )
    col_to_idx = {l: i for i, l in enumerate(cols)}
    for group in mirror_pairs:
        valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
        if len(valid_idx) >= 2:
            group_max = out[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
            for idx in valid_idx:
                out.iloc[:, idx + 1] = group_max

    try:
        tax_df = pd.read_csv(BASE / "taxonomy.csv").set_index("primary_label")
        rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
        for ci, species in enumerate(cols):
            if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
                col_idx = ci + 1
                vals = out.iloc[:, col_idx].to_numpy(np.float32)
                thr = vals.mean() + 0.05
                out.iloc[:, col_idx] = np.where(vals < thr, vals * 0.9, vals)
    except Exception:
        pass

    return out

# Candidate blend settings for the first A2' experiment.
blend_specs = [("base_3way", {"proto": 0.50, "sed": 0.30, "birdnet": 0.20, "effv2s": 0.00})]
if rank_effv2s is not None and rank_birdnet is not None:
    blend_specs.extend([
        ("a2_effv2s_w05", {"proto": 0.47, "sed": 0.28, "birdnet": 0.20, "effv2s": 0.05}),
        ("a2_effv2s_w08", {"proto": 0.45, "sed": 0.27, "birdnet": 0.20, "effv2s": 0.08}),
        ("a2_effv2s_w10", {"proto": 0.44, "sed": 0.26, "birdnet": 0.20, "effv2s": 0.10}),
    ])
elif rank_effv2s is not None:
    blend_specs.extend([
        ("a2_effv2s_twoway_w05", {"proto": 0.57, "sed": 0.38, "birdnet": 0.00, "effv2s": 0.05}),
        ("a2_effv2s_twoway_w10", {"proto": 0.54, "sed": 0.36, "birdnet": 0.00, "effv2s": 0.10}),
    ])

candidates = {}
summary_rows = []
for name, weights in blend_specs:
    pred = weights["proto"] * rank_proto + weights["sed"] * rank_sed
    if rank_birdnet is not None and weights["birdnet"] > 0:
        pred = pred + weights["birdnet"] * rank_birdnet
    if rank_effv2s is not None and weights["effv2s"] > 0:
        pred = pred + weights["effv2s"] * rank_effv2s
    out = apply_postprocess(pred)
    out.to_csv(f"submission_{name}.csv", index=False)
    candidates[name] = out
    summary_rows.append({
        "candidate": name,
        "w_proto": weights["proto"],
        "w_sed": weights["sed"],
        "w_birdnet": weights["birdnet"],
        "w_effv2s": weights["effv2s"],
        "effv2s_active": bool(rank_effv2s is not None),
        "birdnet_active": bool(rank_birdnet is not None),
        "mean_score": float(out[cols].to_numpy(np.float32).mean()),
        "max_score": float(out[cols].to_numpy(np.float32).max()),
    })

if rank_effv2s is not None:
    corr = float(np.corrcoef(rank_effv2s.ravel(), rank_proto.ravel())[0, 1])
else:
    corr = np.nan

pd.DataFrame(summary_rows).assign(proto_effv2s_rank_corr=corr).to_csv("a2prime_blend_summary.csv", index=False)

if rank_effv2s is not None:
    print(f"[A2'] rank corr(effv2s, proto) = {corr:.3f}")
    if corr > 0.85:
        print("[A2'] WARNING: high correlation - likely low blend gain")
    elif corr < 0.30:
        print("[A2'] WARNING: very low correlation - verify mel preprocessing")
    else:
        print("[A2'] OK: useful diversity for blend")
else:
    print("[A2'] EffV2S inactive in this run; falling back to baseline candidates")

default_name = "a2_effv2s_w08" if "a2_effv2s_w08" in candidates else "base_3way"
sub = candidates[default_name].copy()

test_paths = list((BASE / "test_soundscapes").glob("*.ogg"))
if len(test_paths) == 0:
    sample_public = pd.read_csv(BASE / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    sub = sample_public.copy()
    for label in cols:
        sub[label] = template[label]

sub.to_csv(OUT_CSV, index=False)
print(f"A2' blend complete. Default candidate: {default_name}. Saved {OUT_CSV} shape={sub.shape}")


---
## Cell 16 — Submission diagnostics

This final lightweight cell does not change the submission. It only verifies that `submission.csv` exists, has finite values, has one `row_id` column, and contains probability-like numeric columns.

In [ ]:
# Final submission diagnostics: does not alter submission.csv
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

submission_path = Path("submission.csv")
assert submission_path.exists(), "submission.csv was not created. Run the blend cell first."

sub_check = pd.read_csv(submission_path)
prob_cols = [c for c in sub_check.columns if c != "row_id"]

summary = pd.DataFrame({
    "check": [
        "rows",
        "columns",
        "class columns",
        "missing values",
        "min probability",
        "max probability",
        "duplicated row_id",
    ],
    "value": [
        len(sub_check),
        sub_check.shape[1],
        len(prob_cols),
        int(sub_check.isna().sum().sum()),
        float(sub_check[prob_cols].min().min()) if prob_cols else np.nan,
        float(sub_check[prob_cols].max().max()) if prob_cols else np.nan,
        int(sub_check["row_id"].duplicated().sum()) if "row_id" in sub_check.columns else "row_id missing",
    ]
})

display(Markdown("### ✅ Submission diagnostic summary"))
display(summary)

assert "row_id" in sub_check.columns, "row_id column is missing."
assert len(prob_cols) > 0, "No class probability columns found."
assert np.isfinite(sub_check[prob_cols].to_numpy()).all(), "Non-finite values found in probability columns."
assert sub_check[prob_cols].min().min() >= 0.0, "Probability columns contain values below 0."
assert sub_check[prob_cols].max().max() <= 1.0, "Probability columns contain values above 1."

print("submission.csv passed basic diagnostics.")
